# DM Extraction from CHIME H5 Files

Three approaches, in order of reliability:
1. **Metadata search** — dig through every attribute and dataset in each H5 file
2. **Filename parsing** — CHIME files sometimes encode DM or FRB name (look up in catalog)
3. **Brute-force DM search** — trial dedispersion across a grid of DMs, pick the one that maximizes SNR

Output: a `dm_catalog.json` mapping each filename → DM value, ready to paste into the augmentation notebook.

In [ ]:
import numpy as np
import h5py
import os
import glob
import json
import re
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FRB_H5_DIR = "./realtime_data/FRBs/"
h5_files = sorted(glob.glob(os.path.join(FRB_H5_DIR, "*.h5")))
print(f"Found {len(h5_files)} H5 files")

## 1. Deep Metadata Search

Recursively walk every group, dataset, and attribute in each file looking for anything DM-related.

In [ ]:
def deep_search_h5(filepath):
    """
    Recursively walk an H5 file and collect:
    - All attributes (on every group and dataset)
    - All dataset names and shapes
    - Anything that looks like a DM value
    """
    results = {
        "all_attrs": {},
        "datasets": {},
        "dm_candidates": [],
    }
    
    dm_keywords = ["dm", "dispersion", "DM", "dispersion_measure", 
                   "disp_measure", "dm_value", "best_dm", "dm_opt",
                   "dm_snr", "dm_time"]
    
    def visitor(name, obj):
        # Collect attributes
        for attr_name, attr_val in obj.attrs.items():
            full_key = f"{name}/{attr_name}" if name else attr_name
            results["all_attrs"][full_key] = attr_val
            
            # Check if this looks DM-related
            if any(kw.lower() in attr_name.lower() for kw in dm_keywords):
                results["dm_candidates"].append({
                    "source": f"attr:{full_key}",
                    "value": attr_val
                })
        
        # If it's a dataset, record shape and check name
        if isinstance(obj, h5py.Dataset):
            results["datasets"][name] = {
                "shape": obj.shape,
                "dtype": str(obj.dtype)
            }
            # Check if dataset name is DM-related
            if any(kw.lower() in name.lower() for kw in dm_keywords):
                # Try to read it if it's small
                if obj.size <= 100:
                    try:
                        val = obj[()]
                        results["dm_candidates"].append({
                            "source": f"dataset:{name}",
                            "value": val
                        })
                    except:
                        pass
    
    with h5py.File(filepath, 'r') as f:
        # Root-level attributes
        for attr_name, attr_val in f.attrs.items():
            results["all_attrs"][f"/{attr_name}"] = attr_val
            if any(kw.lower() in attr_name.lower() for kw in dm_keywords):
                results["dm_candidates"].append({
                    "source": f"root_attr:{attr_name}",
                    "value": attr_val
                })
        
        f.visititems(visitor)
    
    return results

In [ ]:
# Run deep search on all files
metadata_results = {}
dm_from_metadata = {}
corrupted_files = []

for fpath in h5_files:
    fname = os.path.basename(fpath)
    
    try:
        results = deep_search_h5(fpath)
    except OSError as e:
        print(f"\n{'='*60}")
        print(f"{fname}")
        print(f"{'='*60}")
        print(f"  ❌ CORRUPTED/TRUNCATED FILE — skipping")
        print(f"     Error: {e}")
        fsize_mb = os.path.getsize(fpath) / 1e6
        print(f"     File size: {fsize_mb:.1f} MB (likely incomplete download)")
        corrupted_files.append(fname)
        continue
    
    metadata_results[fname] = results
    
    print(f"\n{'='*60}")
    print(f"{fname}")
    print(f"{'='*60}")
    
    # Show all attributes
    print(f"\n  Attributes ({len(results['all_attrs'])})::")
    for k, v in results["all_attrs"].items():
        # Truncate long values
        v_str = str(v)
        if len(v_str) > 100:
            v_str = v_str[:100] + "..."
        print(f"    {k} = {v_str}")
    
    # Show datasets
    print(f"\n  Datasets ({len(results['datasets'])})::")
    for name, info in results["datasets"].items():
        print(f"    {name}: shape={info['shape']}, dtype={info['dtype']}")
    
    # DM candidates
    if results["dm_candidates"]:
        print(f"\n  ✅ DM candidates found:")
        for cand in results["dm_candidates"]:
            print(f"    {cand['source']} → {cand['value']}")
            try:
                dm_val = float(cand["value"])
                if 1 < dm_val < 10000:  # sanity check: reasonable DM range
                    dm_from_metadata[fname] = dm_val
            except (ValueError, TypeError):
                pass
    else:
        print(f"\n  ❌ No DM found in metadata")

if corrupted_files:
    print(f"\n{'='*60}")
    print(f"⚠️  {len(corrupted_files)} corrupted file(s) skipped:")
    for f in corrupted_files:
        print(f"  - {f}")
    print(f"Consider re-downloading these.")

In [ ]:
print(f"\nDM values found from metadata: {len(dm_from_metadata)}/{len(h5_files)}")
for fname, dm in dm_from_metadata.items():
    print(f"  {fname}: DM = {dm:.2f} pc/cm³")

files_needing_dm = [os.path.basename(f) for f in h5_files 
                      if os.path.basename(f) not in dm_from_metadata
                      and os.path.basename(f) not in corrupted_files]
if files_needing_dm:
    print(f"\nFiles still needing DM ({len(files_needing_dm)}):")
    for f in files_needing_dm:
        print(f"  {f}")

## 2. Filename Parsing

Try to extract FRB names from filenames (e.g. `FRB20180916B`) for catalog lookup.

In [ ]:
# Common FRB naming patterns in CHIME files
frb_name_patterns = [
    r'(FRB\d{8}[A-Z]?)',         # FRB20180916B
    r'(frb\d{8}[a-zA-Z]?)',       # frb20180916b
    r'(FRB_\d{8}[A-Z]?)',         # FRB_20180916B
    r'(\d{8}[A-Z]?)',             # 20180916B (just the date+letter)
]

parsed_names = {}

for fpath in h5_files:
    fname = os.path.basename(fpath)
    stem = Path(fpath).stem  # filename without extension
    
    matched = None
    for pattern in frb_name_patterns:
        match = re.search(pattern, stem)
        if match:
            matched = match.group(1)
            break
    
    # Also check inside file attributes for FRB name
    if matched is None:
        attrs = metadata_results.get(fname, {}).get("all_attrs", {})
        for k, v in attrs.items():
            v_str = str(v)
            for pattern in frb_name_patterns:
                match = re.search(pattern, v_str)
                if match:
                    matched = match.group(1)
                    break
            if matched:
                break
    
    parsed_names[fname] = matched
    status = f"→ {matched}" if matched else "→ no FRB name found"
    print(f"{fname} {status}")

print("\nYou can use these names to look up DMs from the CHIME/FRB catalog:")
print("https://www.chime-frb.ca/catalog")

## 3. Brute-Force DM Search

For files where metadata and filename parsing fail, run a trial dedispersion across a DM grid.

The idea: try many DM values, dedisperse at each, compute the peak SNR of the frequency-collapsed time series. The DM that gives the highest SNR is our best estimate.

**This is the standard approach used in pulsar/FRB search pipelines.**

In [ ]:
# Constants
CHIME_FREQ_LOW  = 400.0   # MHz
CHIME_FREQ_HIGH = 800.0   # MHz
CHIME_TIME_RES  = 0.00098304  # seconds (~0.983 ms), typical CHIME intensity data
K_DM = 4.148808e3  # MHz^2 pc^-1 cm^3 s

def dedisperse_at_dm(power_2d, dm, n_freq, freq_lo=CHIME_FREQ_LOW,
                     freq_hi=CHIME_FREQ_HIGH, time_res=CHIME_TIME_RES):
    """
    Dedisperse a (n_time, n_freq) spectrogram at a given DM.
    Returns the frequency-averaged dedispersed time series.
    """
    n_time = power_2d.shape[0]
    freqs = np.linspace(freq_hi, freq_lo, n_freq)
    delays_s = K_DM * dm * (freqs**-2 - freq_hi**-2)
    delays_bins = np.round(delays_s / time_res).astype(int)
    
    # Pre-allocate and shift
    dedispersed = np.full_like(power_2d, np.nan)
    for i in range(n_freq):
        shift = delays_bins[i]
        if shift == 0:
            dedispersed[:, i] = power_2d[:, i]
        elif shift < n_time:
            dedispersed[:n_time - shift, i] = power_2d[shift:, i]
    
    # Collapse across frequency
    time_series = np.nanmean(dedispersed, axis=1)
    return time_series


def compute_snr(time_series):
    """
    Compute peak SNR of a time series using MAD-based robust noise estimate.
    """
    valid = time_series[~np.isnan(time_series)]
    if len(valid) == 0:
        return 0.0
    median_val = np.median(valid)
    mad = np.median(np.abs(valid - median_val))
    noise_std = 1.4826 * mad
    if noise_std <= 0:
        return 0.0
    return (np.max(valid) - median_val) / noise_std

print("Dedispersion functions ready.")

In [ ]:
def dm_search(power_2d, dm_low=50, dm_high=3000, n_coarse=100, n_fine=50,
              fine_width=50):
    """
    Two-stage DM search:
      1. Coarse grid across full range
      2. Fine grid around the best coarse DM
    
    Parameters:
        power_2d: (n_time, n_freq) spectrogram
        dm_low, dm_high: search range in pc/cm³
        n_coarse: number of coarse trial DMs
        n_fine: number of fine trial DMs around the peak
        fine_width: half-width of fine search range in pc/cm³
        
    Returns:
        best_dm, best_snr, coarse_dms, coarse_snrs, fine_dms, fine_snrs
    """
    n_time, n_freq = power_2d.shape
    
    # --- Stage 1: Coarse search ---
    coarse_dms = np.linspace(dm_low, dm_high, n_coarse)
    coarse_snrs = np.zeros(n_coarse)
    
    for i, dm in enumerate(coarse_dms):
        ts = dedisperse_at_dm(power_2d, dm, n_freq)
        coarse_snrs[i] = compute_snr(ts)
    
    best_coarse_idx = np.argmax(coarse_snrs)
    best_coarse_dm = coarse_dms[best_coarse_idx]
    
    # --- Stage 2: Fine search around peak ---
    fine_lo = max(dm_low, best_coarse_dm - fine_width)
    fine_hi = min(dm_high, best_coarse_dm + fine_width)
    fine_dms = np.linspace(fine_lo, fine_hi, n_fine)
    fine_snrs = np.zeros(n_fine)
    
    for i, dm in enumerate(fine_dms):
        ts = dedisperse_at_dm(power_2d, dm, n_freq)
        fine_snrs[i] = compute_snr(ts)
    
    best_fine_idx = np.argmax(fine_snrs)
    best_dm = fine_dms[best_fine_idx]
    best_snr = fine_snrs[best_fine_idx]
    
    return best_dm, best_snr, coarse_dms, coarse_snrs, fine_dms, fine_snrs

print("DM search function ready.")

In [ ]:
# ============================================================
# CONFIG: adjust these if needed
# ============================================================
BEAM_IDX = 0       # Which beam to use
DM_SEARCH_LOW = 50
DM_SEARCH_HIGH = 3000
N_COARSE = 150     # More = slower but less likely to miss
N_FINE = 80
FINE_WIDTH = 40     # pc/cm³ half-width for fine search

# Optional: subsample time axis for speed on very large files
# Set to None to use full data
MAX_TIME_BINS = None  # e.g. 50000 to cap

In [ ]:
# Run DM search on all files
dm_search_results = {}

for fpath in h5_files:
    fname = os.path.basename(fpath)
    
    # Skip if we already have DM from metadata
    if fname in dm_from_metadata:
        print(f"\n{fname}: DM already known from metadata = {dm_from_metadata[fname]:.2f}, skipping search")
        dm_search_results[fname] = {
            "best_dm": dm_from_metadata[fname],
            "best_snr": None,
            "source": "metadata"
        }
        continue
    
    # Skip corrupted files
    if fname in corrupted_files:
        print(f"\n{fname}: skipping (corrupted)")
        continue
    
    print(f"\n{'='*60}")
    print(f"Searching DM for: {fname}")
    print(f"{'='*60}")
    
    try:
        with h5py.File(fpath, 'r') as f:
            if "tiedbeam_power" not in f:
                print(f"  ⚠️ No tiedbeam_power dataset, skipping")
                continue
            
            ds = f["tiedbeam_power"]
            print(f"  Shape: {ds.shape} (freq, beam, time)")
            
            # Load data: (freq, beam, time) → extract beam → transpose to (time, freq)
            power = ds[:, BEAM_IDX, :].T  # → (time, freq)
        
        n_time, n_freq = power.shape
        print(f"  Working with: ({n_time} time, {n_freq} freq)")
        
        # Optional: subsample for speed
        if MAX_TIME_BINS and n_time > MAX_TIME_BINS:
            # Take a central chunk (burst is likely somewhere in here)
            mid = n_time // 2
            half = MAX_TIME_BINS // 2
            power = power[mid - half:mid + half, :]
            n_time = power.shape[0]
            print(f"  Subsampled to {n_time} time bins")
        
        # Replace NaNs with per-channel median for stability
        for ch in range(n_freq):
            col = power[:, ch]
            nan_mask = np.isnan(col)
            if nan_mask.any():
                col[nan_mask] = np.nanmedian(col)
                power[:, ch] = col
        
        # Run two-stage DM search
        print(f"  Running coarse search ({N_COARSE} trials, DM {DM_SEARCH_LOW}-{DM_SEARCH_HIGH})...")
        best_dm, best_snr, c_dms, c_snrs, f_dms, f_snrs = dm_search(
            power, 
            dm_low=DM_SEARCH_LOW, dm_high=DM_SEARCH_HIGH,
            n_coarse=N_COARSE, n_fine=N_FINE, fine_width=FINE_WIDTH
        )
        
        print(f"  ✅ Best DM = {best_dm:.2f} pc/cm³  (SNR = {best_snr:.1f})")
        
        dm_search_results[fname] = {
            "best_dm": best_dm,
            "best_snr": best_snr,
            "source": "dm_search",
            "coarse_dms": c_dms,
            "coarse_snrs": c_snrs,
            "fine_dms": f_dms,
            "fine_snrs": f_snrs
        }
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        dm_search_results[fname] = {"error": str(e)}

In [ ]:
# Plot DM search curves for all files that went through brute-force search
search_files = [fname for fname, res in dm_search_results.items() 
                if res.get("source") == "dm_search"]

if len(search_files) == 0:
    print("All DMs found from metadata — no search plots to show.")
else:
    n_plots = len(search_files)
    fig, axes = plt.subplots(n_plots, 2, figsize=(14, 4 * n_plots))
    if n_plots == 1:
        axes = axes[np.newaxis, :]
    
    for i, fname in enumerate(search_files):
        res = dm_search_results[fname]
        
        # Coarse search
        axes[i, 0].plot(res["coarse_dms"], res["coarse_snrs"], 'b-', linewidth=0.8)
        axes[i, 0].axvline(res["best_dm"], color='r', linestyle='--', alpha=0.7,
                           label=f'Best DM={res["best_dm"]:.1f}')
        axes[i, 0].set_xlabel("Trial DM (pc/cm³)")
        axes[i, 0].set_ylabel("Peak SNR")
        axes[i, 0].set_title(f"{fname} — Coarse Search")
        axes[i, 0].legend()
        
        # Fine search
        axes[i, 1].plot(res["fine_dms"], res["fine_snrs"], 'g-', linewidth=1.0)
        axes[i, 1].axvline(res["best_dm"], color='r', linestyle='--', alpha=0.7,
                           label=f'Best DM={res["best_dm"]:.2f}')
        axes[i, 1].set_xlabel("Trial DM (pc/cm³)")
        axes[i, 1].set_ylabel("Peak SNR")
        axes[i, 1].set_title(f"{fname} — Fine Search (SNR={res['best_snr']:.1f})")
        axes[i, 1].legend()
    
    plt.tight_layout()
    plt.show()

## 4. Validate: Show Dedispersed Bursts

For each file, show the raw spectrogram and the dedispersed version at the found DM. The burst should collapse into a sharp vertical line when correctly dedispersed.

In [ ]:
valid_files = [fname for fname, res in dm_search_results.items() 
               if "best_dm" in res and "error" not in res
               and fname not in corrupted_files]

if len(valid_files) == 0:
    print("No valid DM results to visualize.")
else:
    n_plots = len(valid_files)
    fig, axes = plt.subplots(n_plots, 3, figsize=(18, 4 * n_plots))
    if n_plots == 1:
        axes = axes[np.newaxis, :]
    
    for i, fname in enumerate(valid_files):
        fpath = os.path.join(FRB_H5_DIR, fname)
        dm = dm_search_results[fname]["best_dm"]
        
        with h5py.File(fpath, 'r') as f:
            power = f["tiedbeam_power"][:, BEAM_IDX, :].T  # → (time, freq)
        
        n_time, n_freq = power.shape
        
        # Dedisperse
        ts = dedisperse_at_dm(power, dm, n_freq)
        
        # Find burst peak
        valid_ts = np.nan_to_num(ts, nan=0.0)
        peak_idx = np.argmax(valid_ts)
        
        # Compute delay for window sizing
        delay_bins = int(np.round(
            K_DM * dm * (CHIME_FREQ_LOW**-2 - CHIME_FREQ_HIGH**-2) / CHIME_TIME_RES
        ))
        margin = max(500, delay_bins * 2)
        t_lo = max(0, peak_idx - margin)
        t_hi = min(n_time, peak_idx + margin)
        
        # Build dedispersed spectrogram for visualization
        freqs = np.linspace(CHIME_FREQ_HIGH, CHIME_FREQ_LOW, n_freq)
        delays_bins = np.round(
            K_DM * dm * (freqs**-2 - CHIME_FREQ_HIGH**-2) / CHIME_TIME_RES
        ).astype(int)
        dd_spec = np.full_like(power, np.nan)
        for ch in range(n_freq):
            shift = delays_bins[ch]
            if shift == 0:
                dd_spec[:, ch] = power[:, ch]
            elif shift < n_time:
                dd_spec[:n_time - shift, ch] = power[shift:, ch]
        
        # Plot 1: Raw spectrogram (zoomed)
        raw_window = power[t_lo:t_hi, :]
        vmin, vmax = np.nanpercentile(raw_window, [5, 95])
        axes[i, 0].imshow(raw_window.T, aspect='auto', origin='lower',
                          vmin=vmin, vmax=vmax, cmap='viridis')
        axes[i, 0].set_title(f"{fname}\nRaw (DM={dm:.1f})")
        axes[i, 0].set_xlabel("Time bin")
        axes[i, 0].set_ylabel("Freq channel")
        
        # Plot 2: Dedispersed spectrogram (zoomed)
        dd_window = dd_spec[t_lo:t_hi, :]
        vmin2, vmax2 = np.nanpercentile(dd_window[~np.isnan(dd_window)], [5, 95]) if np.any(~np.isnan(dd_window)) else (0, 1)
        axes[i, 1].imshow(dd_window.T, aspect='auto', origin='lower',
                          vmin=vmin2, vmax=vmax2, cmap='viridis')
        axes[i, 1].set_title(f"Dedispersed at DM={dm:.1f}")
        axes[i, 1].set_xlabel("Time bin")
        axes[i, 1].set_ylabel("Freq channel")
        axes[i, 1].axvline(peak_idx - t_lo, color='r', linestyle='--', alpha=0.7)
        
        # Plot 3: Dedispersed time series
        axes[i, 2].plot(ts[t_lo:t_hi], 'k-', linewidth=0.5)
        axes[i, 2].axvline(peak_idx - t_lo, color='r', linestyle='--', alpha=0.7)
        axes[i, 2].set_title(f"Dedispersed Time Series (peak SNR)")
        axes[i, 2].set_xlabel("Time bin")
        axes[i, 2].set_ylabel("Power")
    
    plt.tight_layout()
    plt.show()

## 5. Quality Flags

Flag any files where the DM search result looks suspicious.

In [ ]:
print("Quality assessment:\n")

quality_flags = {}

for fname, res in dm_search_results.items():
    if "error" in res:
        quality_flags[fname] = "ERROR"
        print(f"  ❌ {fname}: ERROR — {res['error']}")
        continue
    
    flags = []
    dm = res["best_dm"]
    snr = res.get("best_snr")
    
    # Low SNR: might not be a real detection
    if snr is not None and snr < 5:
        flags.append(f"LOW_SNR ({snr:.1f})")
    
    # DM at edge of search range: might need wider range
    if dm < DM_SEARCH_LOW + 20 or dm > DM_SEARCH_HIGH - 20:
        flags.append(f"DM_AT_EDGE ({dm:.1f})")
    
    # Very low DM: might be RFI not an FRB
    if dm < 100:
        flags.append(f"DM_VERY_LOW ({dm:.1f}) — could be RFI")
    
    # Check if coarse SNR curve has multiple comparable peaks (ambiguous DM)
    if res.get("coarse_snrs") is not None:
        c_snrs = res["coarse_snrs"]
        top_snr = np.max(c_snrs)
        if top_snr > 0:
            # How many DM values give > 80% of peak SNR?
            n_high = np.sum(c_snrs > 0.8 * top_snr)
            if n_high > N_COARSE * 0.15:  # more than 15% of trials
                flags.append("BROAD_PEAK — DM may be poorly constrained")
    
    quality_flags[fname] = flags
    
    if flags:
        print(f"  ⚠️  {fname}: DM={dm:.2f}, SNR={snr}, flags: {', '.join(flags)}")
    else:
        snr_str = f"{snr:.1f}" if snr is not None else "N/A"
        print(f"  ✅ {fname}: DM={dm:.2f}, SNR={snr_str}")

## 6. Export DM Catalog

Save results as JSON and also print the `manual_dm` dict ready to paste into the augmentation notebook.

In [ ]:
# Build final catalog
dm_catalog = {}

for fname, res in dm_search_results.items():
    if "error" in res:
        continue
    dm_catalog[fname] = {
        "dm": round(res["best_dm"], 2),
        "snr": round(res["best_snr"], 2) if res.get("best_snr") is not None else None,
        "source": res["source"],
        "flags": quality_flags.get(fname, [])
    }

# Save JSON
json_path = "./dm_catalog.json"
with open(json_path, 'w') as f:
    json.dump(dm_catalog, f, indent=2)
print(f"Saved DM catalog to {json_path}")

# Print ready-to-paste dict
print("\n" + "="*60)
print("COPY-PASTE THIS INTO THE AUGMENTATION NOTEBOOK:")
print("="*60)
print("manual_dm = {")
for fname, info in dm_catalog.items():
    flag_str = f"  # {', '.join(info['flags'])}" if info['flags'] else ""
    print(f'    "{fname}": {info["dm"]},{flag_str}')
print("}")

if corrupted_files:
    print(f"\n⚠️  Corrupted files excluded ({len(corrupted_files)}):")
    for f in corrupted_files:
        print(f"  - {f}")
    print("Re-download these to include them in training.")

In [ ]:
# Summary plot: DM distribution of our FRB sample
dms = [info["dm"] for info in dm_catalog.values()]
snrs = [info["snr"] for info in dm_catalog.values() if info["snr"] is not None]

if dms:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(dms, bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel("DM (pc/cm³)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"DM Distribution ({len(dms)} FRBs)")
    
    if snrs:
        axes[1].scatter(dms[:len(snrs)], snrs, alpha=0.7, edgecolors='black')
        axes[1].set_xlabel("DM (pc/cm³)")
        axes[1].set_ylabel("Detection SNR")
        axes[1].set_title("DM vs Detection SNR")
        axes[1].axhline(5, color='r', linestyle='--', alpha=0.5, label='SNR=5 threshold')
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No DM values to plot.")

print(f"\n✅ Done! {len(dm_catalog)} files with DM values.")
print(f"Copy the manual_dm dict above into your augmentation notebook.")

In [ ]:
# Export dms and snrs as standalone dicts
dms_dict = {fname: info["dm"] for fname, info in dm_catalog.items()}
snrs_dict = {fname: info["snr"] for fname, info in dm_catalog.items() if info["snr"] is not None}

export = {"dms": dms_dict, "snrs": snrs_dict}

export_path = "./dms_snrs.json"
with open(export_path, "w") as f:
    json.dump(export, f, indent=2)

print(f"Saved to {export_path}")
print(f"  dms:  {len(dms_dict)} entries")
print(f"  snrs: {len(snrs_dict)} entries")
print()
print("To reload:")
print("  import json")
print("  data = json.load(open('./dms_snrs.json'))")
print("  dms, snrs = data['dms'], data['snrs']")